# Producion Agricola (Analisis Exploratorio de Datos)

Una vez realizados los ajustes pertinentes al conjunto de datos principal. Procedemos a realizar el analisis exploratorio de datos.
Donde conoceremos los datos y los iremos preparando para los futuroes modelos planteados.

**Observación** Una de las cosas distintivas con respecto a las etapas pasadas es que si nuestro problema lo requiere conectaremos nuestros datos principales con otras fuentes. Como veremos más adelante

In [1]:
from scripts.config import PRELOADS, DATA_DIR
import os
import pickle
import pandas as pd

# Notemos que ahora estamos obteniendo la información ya normalizada.
with open(PRELOADS['AGROPROD_2'], "rb") as preprocess: 
    dgsiap = pickle.load(preprocess)

dgsiap

Comenzamos con la carga de los conjuntos de datos de Producción Agricola de parte del DGSIAP.

In [15]:
dataset = [file.open(pars=[1, {"low_memory":False, "index_col":0}]) for file in dgsiap.branch]

Recordemos que en esta primera parte procuramos estimar la tasa de éxito de un cultivo en función del clima de la región. Evitando confundir con el rendimiento. La tasa de exito en un cultivo es el cociente entre las hectareas cosechadas y las hectarias sembradas. Para este analisis omitiremos las hectareas siniestradas.

Primero haremos conjunto de datos que considere todos los años.

In [50]:
prodagro = dataset[0]

Como EDA de muestra usaremos un cultivo arbitrario.

In [51]:
# Buscamos todos los cultivos.
crops = prodagro['Nomcultivo'].unique().tolist()
crop_s = crops[20]
crop_s

'Uva'

Tomando la Uva como ejemplo. Vemos los estados que la cosechan 

In [52]:
prod_crop = prodagro[prodagro['Nomcultivo']==crop_s]
crop_rate = prod_crop['Cosechada'].sum() / prod_crop['Sembrada'].sum()
print(f"Tasa de Exito en México de {crop_s}: {crop_rate*100:2f}")

Tasa de Exito en México de Uva: 99.078994


La uva tiene una tasa de exito en Mexico de aproximadamente 94.89%. Haremos lo mismo con por estado.

In [53]:
prod_crop_ent = prod_crop.groupby("Nomestado")[['Sembrada','Cosechada']].sum()
prod_crop_ent['TasaExito'] = prod_crop_ent['Cosechada'] / prod_crop_ent['Sembrada'] * 100
prod_crop_ent

,Sembrada,Cosechada,TasaExito
Nomestado,,,
Aguascalientes,1167.04,1102.04,94.430354
Baja California,4493.49,4488.49,99.888728
Baja California Sur,48.50,10.50,21.649485
Chihuahua,422.83,294.00,69.531490
Coahuila,767.50,676.50,88.143322
Durango,3.00,3.00,100.000000
Guanajuato,511.00,509.00,99.608611
Jalisco,1458.50,1458.50,100.000000
Nayarit,108.00,107.50,99.537037


In [54]:
prod_crop_ent['TasaExito'].mean()

np.float64(90.81786481103282)

Vemos que el porcentaje promedio es mucho menor al porcentaje nacional. Por lo tanto, el porcentaje nacional es una ponderación de los porcentajes estatales.

## Fuente Externa. OPEN METEO API

Para comenzar las conexiones de datos. Revisaremos información provenida por la API de OPEN METEO.

In [ ]:
climate_path = DATA_DIR / "pure" / "OPENMETEO"
climate = [pd.read_csv(climate_path / f) for f in os.listdir(climate_path)]
test = climate[0]

IndexError: list index out of range

In [ ]:
michoacan = [df[df['Idestado'] == 16] for df in dataset]

for k, df in enumerate(michoacan):
    if k==0:
        df_micho = df
    df_micho = pd.concat([df_micho, df])


df_micho.reset_index(drop=True, inplace=True)


In [ ]:
df_micho.head(n=5)

In [ ]:
crops = df_micho['Nomcultivo'].unique().tolist()
len(crops)

In [ ]:
crops_0 = crops[0]
crops_0

In [ ]:
micho_bean = df_micho[df_micho['Nomcultivo']==crops_0]

In [ ]:
micho_bean

In [ ]:
micho_muns = micho_bean['Nommunicipio'].unique().tolist()
len(micho_muns)

In [ ]:
muns_0 = micho_muns[0]

micho_bmun = micho_bean[micho_bean['Nommunicipio']==muns_0]
print(micho_bmun.columns)
micho_bmun.drop(columns=['Idestado','Idcultivo','Idunidadmedida','Idmodalidad'], inplace=True)
micho_bmun

In [ ]:
df = micho_bmun.groupby('Anio')[['Sembrada','Cosechada','Siniestrada']].sum()
df['TasaExito'] = df['Cosechada'] / df['Sembrada']
df

In [ ]:
micho_clima = climate[0]
reg_clima = micho_clima[micho_clima['Nommunicipio']==muns_0]
reg_clima.set_index('Anio',drop=True, inplace=True)
reg_clima

In [ ]:
model = reg_clima.drop(columns=["Idestado", "Idmunicipio", "Nommunicipio","Latitud", "Longitud"])
model['TasaExito']=df['TasaExito']
model

In [ ]:
import matplotlib.pyplot as plt

traindata = model[model["TasaExito"] ==1]
prob = model[model["TasaExito"] <1]
traindata

In [ ]:
prob

In [ ]:
import torch
import numpy as np
from torch import nn
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

batch_size = 5
shuffle = True

features = traindata.iloc[:,:-1]
X = features.to_numpy()
target = features.iloc[:, -1].to_numpy()

dataset = TensorDataset(torch.Tensor(X), torch.Tensor(target))

dataloader=DataLoader(dataset, batch_size=batch_size, shuffle=1)

m = nn.Linear(in_features=7,out_features=10)
x = torch.rand(1, 7)
x1 = nn.Sigmoid()(m(x))
m1 = nn.Linear(in_features=10, out_features=1)
y = m1(x1)

crit = nn.MSELoss()

loss = crit(y.double(), torch.tensor([1]).double())
loss.backward()

optimizer = torch.optim.SGD(m.parameters(), lr=0.001)

optimizer.step()

y

In [ ]:
import torch
import numpy as np
from torch import nn
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

batch_size = 5
shuffle = True

features = traindata.iloc[:,:-1]
X = features.to_numpy()
target = traindata.iloc[:, -1].to_numpy()

dataset = TensorDataset(torch.Tensor(X), torch.Tensor(target))

dataloader=DataLoader(dataset, batch_size=batch_size, shuffle=1)

agromodel = nn.Sequential(nn.Linear(in_features=7,out_features=10),
nn.Linear(in_features=10, out_features=1))


crit = nn.MSELoss()
optimizer = torch.optim.SGD(agromodel.parameters(), lr=0.001)

for epoch in range(1_000):
    for data in dataloader:
        optimizer.zero_grad()
        feature, target = data
        pred = agromodel(feature)
        loss = crit(pred, target)
        loss.backward()
        optimizer.step()



In [ ]:
test = prob.iloc[:,:-1].to_numpy()
test

In [ ]:
uq = torch.Tensor(traindata.loc[2003].iloc[:-1].to_numpy())

In [ ]:
agromodel(uq).detach().numpy()